In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

class ReflectiveAgent:
    def __init__(self, max_iterations=3):
        self.max_iterations = max_iterations
        self.memory = []  # 反思記錄
    
    def execute(self, task: str) -> str:
        """執行任務，帶有反思迴圈"""
        # 組合過去的反思記錄
        context = self._build_context(task)
        
        for i in range(self.max_iterations):
            # Step 1: Actor 執行
            result = self._act(context)
            
            # Step 2: Evaluator 評估
            score, feedback = self._evaluate(task, result)
            
            if score >= 4.0:  # 品質達標，結束迴圈
                return result
            
            # Step 3: Self-Reflection 反思
            reflection = self._reflect(result, feedback)
            
            # Step 4: 存入記憶
            self.memory.append({
                "iteration": i + 1,
                "score": score,
                "reflection": reflection
            })
            
            # 更新 context 供下一輪使用
            context = self._build_context(task)
        
        return result  # 達到最大迭代次數，回傳最後結果
    
    def _act(self, context: str) -> str:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=4096,
            messages=[{"role": "user", "content": context}]
        )
        return response.content[0].text
    
    def _evaluate(self, task, result):
        prompt = f"""評估以下產出的品質（1-5 分）：
任務：{task}
產出：{result}

回傳 JSON：{{"score": 數字, "feedback": "具體回饋"}}"""
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        data = json.loads(response.content[0].text)
        return data["score"], data["feedback"]
    
    def _reflect(self, result, feedback):
        prompt = f"""你的產出收到以下回饋：{feedback}

請反思：
1. 具體犯了什麼錯？
2. 根本原因是什麼？
3. 下次應該怎麼做？

用一段話總結。"""
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    
    def _build_context(self, task):
        ctx = f"任務：{task}\n\n"
        if self.memory:
            ctx += "過去的反思記錄（請參考這些經驗避免重複犯錯）：\n"
            for m in self.memory:
                ctx += f"- 第{m['iteration']}次嘗試（{m['score']}分）：{m['reflection']}\n"
        return ctx

# 使用範例
agent = ReflectiveAgent(max_iterations=3)
result = agent.execute("分析台灣中小企業 2026 年的 AI 導入現況")
print(result)